In [286]:
from sklearn.linear_model import LinearRegression
from bokeh.models import HoverTool 
from bokeh.io import export_svg
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.signal import peak_widths, find_peaks, peak_prominences, medfilt
import datetime
from sklearn.decomposition import PCA
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
bokeh.io.output_notebook()
pn.extension()
import plotly.graph_objects as go
from bokeh.io import export_png

Loading BokehJS ...

In [212]:
def filter_df(df, condition):
    new_df = df[df['condition'] == condition].copy()
    new_df.reset_index(inplace = True)
    return new_df

# Import csv

In [213]:
df = pd.read_csv('/home/jxu/hand_orthosis_ws/src/trakstar_ros/p1_parsed_data.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head(3)


,condition,raw_time,time,futek,button,motor_position,trakstar_01,trakstar_02,time_unitless,mcp_angle,pip_angle,finger_angle
0,passive_start_calibration,1706025956634248459,2024-01-23 16:05:56.63,0.0,NaN,0.0,[[-0.00931568 -0.909313 -0.41600851 0.03003...,[[ 0.24198414 -0.95177151 -0.1886125 0.08599...,3,0.0,0.0,0.0
1,passive_start_calibration,1706025956671610541,2024-01-23 16:05:56.67,0.0,NaN,0.0,[[-0.00966663 -0.90900383 -0.41667564 0.03003...,[[ 0.24087849 -0.95229913 -0.18736042 0.08609...,6,0.0,0.0,0.0
2,passive_start_calibration,1706025956734066089,2024-01-23 16:05:56.73,0.0,NaN,0.0,[[-0.01131846 -0.90777113 -0.4193131 0.03032...,[[ 0.23855163 -0.95323916 -0.18554845 0.08622...,11,0.0,0.0,0.0


# Separate big dataframe to smaller dataframes

In [214]:
active_rorcr_df = filter_df(df, 'active_rorcr')
active_cube_df = filter_df(df, 'active_cube')
passive_start_df = filter_df(df, 'passive_start')
passive_end_df = filter_df(df, 'passive_end')

# Manually extract opens

In [215]:
def plot_time_condition(df_, condition):
    hover = HoverTool(tooltips=[
        ("index", "$index"),
    ])
    p = bokeh.plotting.figure(
            width=1000,
            height=600,
            x_axis_label='Time',
            y_axis_label='Force (N)',
            title = 'Force vs. Distance',
            #y_range = [-2, 20],
            tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap']
        )

    p.circle(df_.index, df_[condition], color = 'blue', size = 5)

    #p.legend.location='top_left'
    #p.legend.click_policy='hide'
    p.title.text_font_size = '20pt'
    p.xaxis.axis_label_text_font_size = '15pt'
    p.yaxis.axis_label_text_font_size = '15pt'
    p.select(name='index')
    bokeh.io.show(p)

In [216]:
p = plot_time_condition(active_rorcr_df, 'motor_position')

In [255]:
active_rorcr_opens = [[1856, 1934], [2505, 2575], [3244, 3305]]
active_cube_opens = [[626, 680], [1003, 1053], [1946, 1990], [3327, 3371], [4834, 4878], [5978, 6028], [6178, 6219], [6549, 6590], [6853, 6892], [7260, 7303], [7550, 7590], [9115, 9161], [9322, 9366], [9788, 9831], [10697, 10746], [11063, 11108], [16324, 16369], [17175, 17220], [17741, 17782], [18209, 18249], [18582, 18602], [18661, 18699], [23887, 23910], [24319, 24342], [24440, 24504], [26049, 26097], [26274, 26304]]
passive_start_opens = [[66, 110], [199, 236], [305, 348], [425, 462], [535, 578], [658, 702], [769, 810], [875, 920], [994, 1030], [1090, 1135], [1204, 1249]]
passive_end_opens = [[225, 266], [319, 351], [414, 455], [501, 541], [593, 632], [705, 743], [805, 844], [896, 936], [995, 1044], [1099, 1139]]

In [256]:
df_ = pd.DataFrame(np.array(passive_start_opens), columns = ['open_i', 'open_f'])
df_['condition'] = 'passive_start'

for i, j in zip([active_rorcr_opens, active_cube_opens, passive_end_opens], ['active_rorcr', 'active_cube', 'passive_end']):
    sub_df = pd.DataFrame(np.array(i), columns = ['open_i', 'open_f'])
    sub_df['condition'] = j

    df_ = pd.concat([df_, sub_df], axis = 0)

df_.reset_index(inplace=True)
df_.tail(3)


,index,open_i,open_f,condition
48,7,896,936,passive_end
49,8,995,1044,passive_end
50,9,1099,1139,passive_end


In [257]:
df_['m'] = 0
df_['b'] = 0
df_.head(3)

,index,open_i,open_f,condition,m,b
0,0,66,110,passive_start,0,0
1,1,199,236,passive_start,0,0
2,2,305,348,passive_start,0,0


In [258]:
slice(df_['open_i'][0], df_['open_f'][0])

slice(66, 110, None)

In [259]:
mega_df = pd.concat([active_rorcr_df, active_cube_df, passive_start_df, passive_end_df])
mega_df.reset_index(inplace=True, drop=False)
mega_df.drop(columns = ['index'], inplace=True)
mega_df.rename(columns = {'level_0': 'self_index'}, inplace = True)
mega_df.tail(3)

,self_index,condition,raw_time,time,futek,button,motor_position,trakstar_01,trakstar_02,time_unitless,mcp_angle,pip_angle,finger_angle
33515,1204,passive_end,1706029364433564171,2024-01-23 17:02:44.43,0.000000,NaN,0.189,[[-0.10858117 -0.67491537 -0.72986257 0.01373...,[[-0.99040069 -0.13335367 -0.03637693 0.03624...,111045,8.424636,68.063246,76.487882
33516,1205,passive_end,1706029364446243606,2024-01-23 17:02:44.45,0.287921,NaN,0.189,[[-0.10852164 -0.67477073 -0.73000515 0.01364...,[[-0.99044893 -0.13296883 -0.03647207 0.03624...,111046,8.436597,68.073383,76.509980
33517,1206,passive_end,1706029364470999959,2024-01-23 17:02:44.47,0.000000,NaN,0.189,[[-0.10826247 -0.67444804 -0.73034175 0.01372...,[[-0.99052596 -0.13237439 -0.03654235 0.03625...,111048,8.466852,68.076992,76.543844


In [261]:
opens = []
c = 0
for condition in ['passive_start', 'active_rorcr', 'active_cube', 'passive_end']:
    for i in range(len(filter_df(df_, condition))):
        sdf = pd.DataFrame(filter_df(mega_df, condition).iloc[slice(filter_df(df_, condition).at[i, 'open_i'], filter_df(df_, condition).at[i, 'open_f'])].copy())
        Xx = sdf['motor_position'].values.reshape(-1, 1)
        yy = sdf['futek'].values
        model = LinearRegression()
        model.fit(Xx, yy)
        df_.at[c, 'm'] = model.coef_
        df_.at[c, 'b'] = model.intercept_
    
        opens.append(sdf)
        c += 1

final_df = pd.concat(opens, axis = 0)
final_df = final_df[['condition', 'futek', 'motor_position', 'mcp_angle', 'pip_angle']]
final_df.reset_index(inplace=True, drop=True)
final_df.head(4)

,condition,futek,motor_position,mcp_angle,pip_angle
0,passive_start,0.287921,0.077,42.468609,30.972609
1,passive_start,0.000000,0.077,42.468609,30.972609
2,passive_start,0.000000,0.077,42.479387,30.961831
3,passive_start,0.000000,0.077,42.487968,30.953250


In [302]:
colors = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        x_axis_label='Motor Position (mm)',
        y_axis_label='Force (N)',
        title = 'Force vs. Distance',
    )
t = np.linspace(0, 30, 100)
w = 12
a = 0.6
p.circle(x='motor_position', y='futek', source = filter_df(final_df, 'active_cube'), color = colors[2], legend_label = 'active_cube', size = w, alpha = 0.4)
p.circle(x='motor_position', y='futek', source = filter_df(final_df, 'passive_start'), color = colors[0], legend_label = 'passive_start', size = w, alpha = 0.7)
p.circle(x='motor_position', y='futek', source = filter_df(final_df, 'active_rorcr'), color = colors[1], legend_label = 'active_rorcr', size = w, alpha = 0.7)
p.circle(x='motor_position', y='futek', source = filter_df(final_df, 'passive_end'), color = colors[3], legend_label = 'passive_end', size = w, alpha = 0.7)


p.legend.location='top_left'
p.legend.click_policy='hide'
p.title.text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '15pt'
p.yaxis.axis_label_text_font_size = '15pt'

bokeh.io.show(p)
export_png(p, filename="strain_plot.png")


'/home/jxu/hand_orthosis_ws/src/trakstar_ros/strain_plot.png'

In [284]:
final_df.head()

,condition,futek,motor_position,mcp_angle,pip_angle
0,passive_start,0.287921,0.077,42.468609,30.972609
1,passive_start,0.000000,0.077,42.468609,30.972609
2,passive_start,0.000000,0.077,42.479387,30.961831
3,passive_start,0.000000,0.077,42.487968,30.953250
4,passive_start,0.000000,0.077,42.492808,30.948410


In [ ]:
colors = bokeh.palettes.d3['Category10'][10]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        x_axis_label='Motor Position (mm)',
        y_axis_label='Force (N)',
        title = 'Force vs. Distance',
    )
t = np.linspace(0, 30, 100)
w = 10
a = 0.6
p.circle(x='motor_position', y='futek', source = filter_df(final_df, 'passive_start'), color = colors[0], legend_label = 'passive_start', size = w, alpha = a)
p.circle(x='motor_position', y='futek', source = filter_df(final_df, 'active_rorcr'), color = colors[1], legend_label = 'active_rorcr', size = w, alpha = a)


p.legend.location='top_left'
p.legend.click_policy='hide'
p.title.text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '15pt'
p.yaxis.axis_label_text_font_size = '15pt'

bokeh.io.show(p)

In [263]:
from bokeh.models import CDSView, ColumnDataSource, IndexFilter

In [295]:
from bokeh.transform import factor_cmap
colors = bokeh.palettes.d3['Category10'][4]
p = bokeh.plotting.figure(
        width=1000,
        height=600,
        x_axis_label='Experiment Time',
        y_axis_label='Stiffness',
        title = 'Stiffness vs. Time',
    )

source = ColumnDataSource(data = dict(x=df_.index, y = df_['m'], col = df_['condition'], label = df_['condition']))

cols = factor_cmap('col', palette= colors, factors=df_['condition'].unique())
p.circle(x='x', y='y', source = source, color = cols, legend_field = 'label', size = 15)
p.segment(x0='x', y0 = 0, x1 = 'x', y1='y', source = source, color = cols, legend_field = 'label')

p.legend.location='bottom_right'
p.legend.click_policy='hide'
p.title.text_font_size = '20pt'
p.xaxis.axis_label_text_font_size = '15pt'
p.yaxis.axis_label_text_font_size = '15pt'

bokeh.io.show(p)
export_png(p, filename="stiffness_plot.png")

'/home/jxu/hand_orthosis_ws/src/trakstar_ros/stiffness_plot.png'

In [265]:
df_

,index,open_i,open_f,condition,m,b
0,0,66,110,passive_start,0.077883,-0.313108
1,1,199,236,passive_start,0.100732,-0.489811
2,2,305,348,passive_start,0.093590,-0.443544
3,3,425,462,passive_start,0.097499,-0.477876
4,4,535,578,passive_start,0.099758,-0.519920
5,5,658,702,passive_start,0.095193,-0.457902
6,6,769,810,passive_start,0.104147,-0.507899
7,7,875,920,passive_start,0.104566,-0.509258
8,8,994,1030,passive_start,0.109357,-0.602197
9,9,1090,1135,passive_start,0.100611,-0.503653


In [240]:
from bokeh.layouts import gridplot
from bokeh.models import CDSView, ColumnDataSource, IndexFilter
from bokeh.plotting import figure, show

# create ColumnDataSource from a dict
source = ColumnDataSource(data=df_)

# create a view using an IndexFilter with the index positions [0, 2, 4]
view = CDSView(filter=GroupFilter(column_name = 'condition', group = 'active_rorcr'))
view1 = CDSView(filter=GroupFilter(column_name = 'condition', group = 'active_cube'))

# create a first plot with all data in the ColumnDataSource
p = figure(height=300, width=300)
p.circle(x="index", y="m", size=10, hover_color="red", source=source, view=view, legend_field = 'condition')
p.circle(x="index", y="m", size=10, hover_color="red", source=source, view=view1, legend_field = 'condition')

# show both plots next to each other in a gridplot layout
show(p)

In [292]:
def get_mcp_pip_plot(df, condition):
    x = 'time_unitless'
    y = 'mcp_angle'
    z = 'pip_angle'

    p = bokeh.plotting.figure(
            width=800,
            height=600,
            x_axis_label='time units',
            y_axis_label='angle (degrees)',
            title = condition + ' MCP and PIP angles',
            y_range = [-45, 200]
        )

    p.circle(filter_df(df, condition)[x], filter_df(df, condition)[y], color = 'navy', legend_label = y, alpha = 0.75)
    p.circle(filter_df(df, condition)[x], filter_df(df, condition)[z], color = 'red', legend_label = z, alpha = 0.75)
    #p.circle(filter_df(df, condition)[x], filter_df(df, condition)['finger_angle'], color = 'grey', legend_label = 'finger_angle', alpha= 0.3)

    return p

In [289]:
mega_df

,self_index,condition,raw_time,time,futek,button,motor_position,trakstar_01,trakstar_02,time_unitless,mcp_angle,pip_angle,finger_angle
0,0,active_rorcr,1706027804383511035,2024-01-23 16:36:44.38,0.000000,NaN,0.210,[[-0.04082787 -0.6852673 -0.72714635 0.01977...,[[-0.96679183 0.17955705 -0.18185934 0.03036...,9353,22.468436,80.446743,102.915179
1,1,active_rorcr,1706027804396270950,2024-01-23 16:36:44.40,0.000000,NaN,0.210,[[-0.04082787 -0.6852673 -0.72714635 0.01977...,[[-0.96679183 0.17955705 -0.18185934 0.03036...,9354,22.468436,80.446743,102.915179
2,2,active_rorcr,1706027804421039215,2024-01-23 16:36:44.42,0.000000,NaN,0.210,[[-0.04082787 -0.6852673 -0.72714635 0.01977...,[[-0.96679183 0.17955705 -0.18185934 0.03036...,9356,22.468436,80.446743,102.915179
3,3,active_rorcr,1706027804471029087,2024-01-23 16:36:44.47,0.000000,NaN,0.210,[[-0.04064594 -0.6852333 -0.72718858 0.01977...,[[-0.96678417 0.1794003 -0.18205466 0.03036...,9360,22.474558,80.430717,102.905275
4,4,active_rorcr,1706027804508684239,2024-01-23 16:36:44.51,0.000000,NaN,0.210,[[-0.04047471 -0.68536249 -0.72707638 0.01977...,[[-0.96678417 0.1794003 -0.18205466 0.03036...,9363,22.466818,80.438458,102.905275
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33513,1202,passive_end,1706029364358475248,2024-01-23 17:02:44.36,0.000000,NaN,0.189,[[-0.10944335 -0.67451429 -0.73010453 0.01390...,[[-0.99022306 -0.1347167 -0.03618991 0.03626...,111039,8.442736,67.966139,76.408875
33514,1203,passive_end,1706029364396036052,2024-01-23 17:02:44.40,0.000000,NaN,0.189,[[-0.10943881 -0.6746905 -0.72994238 0.01378...,[[-0.99030915 -0.13411894 -0.03605408 0.03623...,111042,8.429139,68.015357,76.444496
33515,1204,passive_end,1706029364433564171,2024-01-23 17:02:44.43,0.000000,NaN,0.189,[[-0.10858117 -0.67491537 -0.72986257 0.01373...,[[-0.99040069 -0.13335367 -0.03637693 0.03624...,111045,8.424636,68.063246,76.487882
33516,1205,passive_end,1706029364446243606,2024-01-23 17:02:44.45,0.287921,NaN,0.189,[[-0.10852164 -0.67477073 -0.73000515 0.01364...,[[-0.99044893 -0.13296883 -0.03647207 0.03624...,111046,8.436597,68.073383,76.509980


In [293]:
for i in mega_df['condition'].unique():
    p = get_mcp_pip_plot(mega_df, i)
    p.title.text_font_size = '20pt'
    p.legend.label_text_font_size = '20pt'
    p.xaxis.axis_label_text_font_size = '20pt'
    p.yaxis.axis_label_text_font_size = '20pt'
    p.yaxis.major_label_text_font_size = '18pt'
    #p.legend.click_policy = 'hide'
    bokeh.io.show(p)
    export_png(p, filename="%s_angle_plot.png"%i)